In [9]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display

# ── Config ────────────────────────────────────────────────
DATA_DIR  = Path('./food_dataset')
VALID_EXT = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

# ── Scan dataset ──────────────────────────────────────────
rows = []
for cls_dir in sorted(DATA_DIR.iterdir()):
    if cls_dir.is_dir():
        count = len([f for f in cls_dir.iterdir() if f.suffix.lower() in VALID_EXT])
        rows.append({'Class': cls_dir.name, 'Images': count})

df = pd.DataFrame(rows).sort_values('Images', ascending=False).reset_index(drop=True)
df.index += 1

# ── Summary ───────────────────────────────────────────────
print(f"Total classes : {len(df)}")
print(f"Total images  : {df['Images'].sum():,}")
print(f"Min images    : {df['Images'].min()}  →  {df.loc[df['Images'].idxmin(), 'Class']}")
print(f"Max images    : {df['Images'].max()}  →  {df.loc[df['Images'].idxmax(), 'Class']}")
print(f"Mean images   : {df['Images'].mean():.1f}")
print()

# ── Colour-coded table ────────────────────────────────────
def color_rows(val):
    if val < 80:  return 'background-color: #ffcccc'  # red   → likely remove
    if val < 130: return 'background-color: #fff3cc'  # yellow → discuss
    return 'background-color: #ccffcc'                 # green  → keep

styled = df.style.map(color_rows, subset=['Images'])
display(styled)

Total classes : 77
Total images  : 16,101
Min images    : 140  →  mixed_veg_curry
Max images    : 562  →  chole_bhature
Mean images   : 209.1



,Class,Images
1,chole_bhature,562
2,biryani,398
3,halwa,388
4,paratha,388
5,naan,375
6,khichdi,369
7,dosa,367
8,rajma_chawal,366
9,pulao,357
10,kadhi,252


In [8]:
import os
import shutil
from pathlib import Path

DATA_DIR  = Path('./food_dataset')
VALID_EXT = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

# ── Merge map: source folders → target folder ─────────────
MERGES = {
    'biryani'      : ['chicken_biryani'],
    'pulao'        : ['veg_pulao'],
    'naan'         : ['butter_naan'],
    'paratha'      : ['aloo_paratha'],
    'chole_bhature': ['chole', 'bhature'],
    'khichdi'      : ['dal_rice'],
    'dosa'         : ['masala_dosa'],
    'kadhi'        : ['gujarati_kadhi'],
    'halwa'        : ['gajar_halwa'],
    'rajma_chawal' : ['rajma'],
}

# ── Classes to fully remove ────────────────────────────────
REMOVE_CLASSES = [
    "hyderabadi_thali",
    "udupi_thali",
    "tamil_nadu_thali",
    "goan_thali",
    "kashmiri_wazwan",
    "south_indian_thali",
    "kerala_sadya",
    "punjabi_thali",
    "bengali_thali",
    "andhra_thali",
    "maharashtrian_thali",
    "karnataka_thali",
    "rajasthani_thali",
    "lemon_rice",
    "jeera_rice",
    "curd_rice",
    "sambar_rice",
    "kulcha",
]

# ── Correction: dal_makhani stays, remove from list ───────
REMOVE_CLASSES = [c for c in REMOVE_CLASSES if c != 'dal_makhani']

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — MERGE
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 55)
print("STEP 1 — MERGING CLASSES")
print("=" * 55)

for target, sources in MERGES.items():
    target_dir = DATA_DIR / target
    if not target_dir.exists():
        print(f"  ⚠️  Target folder not found: {target}")
        continue

    for src in sources:
        src_dir = DATA_DIR / src
        if not src_dir.exists():
            print(f"  ⚠️  Source folder not found: {src}")
            continue

        imgs = [f for f in src_dir.iterdir() if f.suffix.lower() in VALID_EXT]
        moved = 0

        for img in imgs:
            # Rename to avoid filename collision
            new_name = f"{src}__{img.name}"
            dst      = target_dir / new_name
            shutil.move(str(img), str(dst))
            moved += 1

        # Remove now-empty source folder
        shutil.rmtree(src_dir)
        print(f"  ✅ Merged '{src}' → '{target}'  ({moved} images moved)")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — REMOVE
# ─────────────────────────────────────────────────────────────────────────────
print()
print("=" * 55)
print("STEP 2 — REMOVING CLASSES")
print("=" * 55)

for cls in REMOVE_CLASSES:
    cls_dir = DATA_DIR / cls
    if cls_dir.exists():
        count = len([f for f in cls_dir.iterdir() if f.suffix.lower() in VALID_EXT])
        shutil.rmtree(cls_dir)
        print(f"  🗑️  Removed '{cls}'  ({count} images deleted)")
    else:
        print(f"  ⚠️  Not found (already removed?): {cls}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — FINAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print()
print("=" * 55)
print("STEP 3 — FINAL DATASET SUMMARY")
print("=" * 55)

import pandas as pd
from IPython.display import display

rows = []
for cls_dir in sorted(DATA_DIR.iterdir()):
    if cls_dir.is_dir():
        count = len([f for f in cls_dir.iterdir() if f.suffix.lower() in VALID_EXT])
        rows.append({'Class': cls_dir.name, 'Images': count})

df = pd.DataFrame(rows).sort_values('Images', ascending=False).reset_index(drop=True)
df.index += 1

print(f"  Total classes : {len(df)}")
print(f"  Total images  : {df['Images'].sum():,}")
print(f"  Min images    : {df['Images'].min()}  →  {df.loc[df['Images'].idxmin(), 'Class']}")
print(f"  Max images    : {df['Images'].max()}  →  {df.loc[df['Images'].idxmax(), 'Class']}")
print(f"  Mean images   : {df['Images'].mean():.1f}")
print()

def color_rows(val):
    if val < 80:  return 'background-color: #ffcccc'
    if val < 130: return 'background-color: #fff3cc'
    return 'background-color: #ccffcc'

display(df.style.map(color_rows, subset=['Images']))

STEP 1 — MERGING CLASSES
  ⚠️  Source folder not found: chicken_biryani
  ⚠️  Source folder not found: veg_pulao
  ⚠️  Source folder not found: butter_naan
  ⚠️  Source folder not found: aloo_paratha
  ⚠️  Source folder not found: chole
  ⚠️  Source folder not found: bhature
  ⚠️  Source folder not found: dal_rice
  ⚠️  Source folder not found: masala_dosa
  ⚠️  Source folder not found: gujarati_kadhi
  ⚠️  Source folder not found: gajar_halwa
  ⚠️  Source folder not found: rajma

STEP 2 — REMOVING CLASSES
  ⚠️  Not found (already removed?): hyderabadi_thali
  ⚠️  Not found (already removed?): udupi_thali
  ⚠️  Not found (already removed?): tamil_nadu_thali
  ⚠️  Not found (already removed?): goan_thali
  ⚠️  Not found (already removed?): kashmiri_wazwan
  ⚠️  Not found (already removed?): south_indian_thali
  ⚠️  Not found (already removed?): kerala_sadya
  ⚠️  Not found (already removed?): punjabi_thali
  ⚠️  Not found (already removed?): bengali_thali
  ⚠️  Not found (already remove

,Class,Images
1,chole_bhature,562
2,biryani,398
3,halwa,388
4,paratha,388
5,naan,375
6,khichdi,369
7,dosa,367
8,rajma_chawal,366
9,pulao,357
10,kadhi,252
